# Analisis del Impacto Ambiental del Monocultivo de Aguacate en Michoacan
## HackODS UNAM 2026 | ODS 15: Vida de Ecosistemas Terrestres

Este estudio analiza la relacion entre la expansion del aguacate y la degradacion forestal en Michoacan, utilizando datos del **Censo Agropecuario 2022 (INEGI)** y **Global Forest Watch (GFW)**.

### Glosario de Variables y Origen de Datos GFW
| Variable | Atributo Origen (GFW) | Definicion Tecnica |
| :--- | :--- | :--- |
| **ha_cobertura_2010** | `umd_tree_cover_extent_2010__ha` | Superficie forestal base al inicio de la decada. |
| **ha_perdida_agricola**| `loss_area_ha (Permanent Agriculture)` | Perdida de bosque causada directamente por agricultura perenne (Aguacate). |
| **ha_perdida_primaria**| `umd_tree_cover_loss__ha (Primary Forest)` | Perdida de bosque virgen (primario), el ecosistema mas valioso. |
| **impact_ratio** | `SUPSEM_AGCA / SUP_CARTOG_AG` | **Indice de Invasion**: Relacion entre siembra declarada y suelo agricola disponible. |

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import pearsonr

DATA_DIR = "data/procesados/"
COLORS = {'verde': '#2E8B57', 'cafe': '#8B4513', 'rojo': '#D32F2F'}

print("INFO: Cargando datos procesados de GFW e INEGI...")

# 1. El Motor de la Deforestacion: Agricultura Permanente
Utilizamos datos de GFW filtrados por el motor de perdida **'Permanent Agriculture'**, que en Michoacan es ocupado predominantemente por el aguacate.

In [ ]:
# Carga de serie historica de perdida agricola
df_agri = pd.read_csv(os.path.join(DATA_DIR, "gfw_historico_agricola_mich.csv"))
df_agri = df_agri[df_agri['loss_year'] >= 2012]

# Datos de produccion (Serie real 2012-2022)
years = np.arange(2012, 2025)
prod_vals = {2012: 1.1, 2017: 1.9, 2019: 2.1, 2022: 3.42}
production = np.interp(years, list(prod_vals.keys()), list(prod_vals.values()))

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.set_xlabel('Anio')
ax1.set_ylabel('Produccion Aguacate (Millones Ton)', color=COLORS['verde'])
ax1.plot(years, production, color=COLORS['verde'], marker='o', linewidth=3)
ax1.tick_params(axis='y', labelcolor=COLORS['verde'])

ax2 = ax1.twinx()
ax2.set_ylabel('Perdida por Agricultura Permanente (Ha)', color=COLORS['cafe'])
ax2.bar(df_agri['loss_year'], df_agri['loss_area_ha'], color=COLORS['cafe'], alpha=0.3, label='Perdida GFW (Agri)')
ax2.tick_params(axis='y', labelcolor=COLORS['cafe'])

plt.title('Impacto Directo: Produccion vs Perdida Forestal por Agricultura (2012-2024)')
fig.tight_layout()
plt.show()

# 2. Radiografia Municipal: Invasión de la Frontera Agricola

Esta visualizacion muestra como el exito economico (tamanio de burbuja) se traduce en un riesgo de invasion forestal (color rojo).

In [ ]:
df_final = pd.read_csv(os.path.join(DATA_DIR, "dataset_maestro_ods15.csv"))

fig_bubble = px.scatter(
    df_final, 
    x="ha_aguacate", 
    y="pct_bosque", 
    size="ton_aguacate", 
    color="impact_ratio",
    hover_name="municipio",
    title="Analisis de Burbujas: Produccion vs Riesgo de Invasion Forestal",
    labels={
        'ha_aguacate': 'Superficie Aguacate (Ha)', 
        'pct_bosque': '% de Salud Forestal', 
        'impact_ratio': 'Indice de Invasion'
    },
    color_continuous_scale=px.colors.diverging.RdYlGn[::-1],
    range_color=[0.2, 1.2],
    template="plotly_dark"
)

fig_bubble.update_traces(marker=dict(line=dict(width=1, color='white')))
fig_bubble.show()

# 3. El Costo Irreparable: Perdida de Bosque Primario
Michoacan ha perdido miles de hectareas de bosque virgen. Estos ecosistemas son criticos para la biodiversidad y la captura de carbono.

In [ ]:
df_primary = pd.read_csv(os.path.join(DATA_DIR, "gfw_historico_primario_mich.csv"))
df_primary = df_primary[df_primary['anio'] >= 2012]

fig_pri = px.area(
    df_primary, x="anio", y="ha_perdida_primaria",
    title="Perdida de Bosque Primario en Michoacan (2012-2024)",
    labels={'anio': 'Anio', 'ha_perdida_primaria': 'Hectareas de Bosque Virgen Perdidas'},
    template="plotly_white",
    color_discrete_sequence=[COLORS['rojo']]
)
fig_pri.show()